# Notebook 00 - Monitoramento do Banco

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

DB_USER = os.getenv("POSTGRES_USER", "fs_user")
DB_PASS = os.getenv("POSTGRES_PASSWORD", "fs_pass")
DB_HOST = os.getenv("POSTGRES_HOST", "postgres")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")
DB_NAME = os.getenv("POSTGRES_DB", "fs_mix")

conn_str = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(conn_str)
print("Conectando em:", f"{DB_HOST}:{DB_PORT}/{DB_NAME}")


Conectando em: postgres:5432/fs_mix


In [2]:
def q(sql: str):
    return pd.read_sql_query(text(sql), engine)


## 1) Tabelas

In [3]:
q("SELECT table_schema, table_name FROM information_schema.tables WHERE table_schema IN ('bronze','silver') ORDER BY table_schema, table_name")


,table_schema,table_name
0,bronze,file_registry
1,bronze,movimentacoes_raw
2,bronze,produtos_servicos_raw
3,silver,dim_produtos_servicos
4,silver,fato_movimentacoes


## 2) Amostras e checks rapidos (status de processamento)


In [4]:
q("""
SELECT
    file_name,
    file_type,
    status,
    rows_loaded,
    processed_at,
    error_message
FROM bronze.file_registry
ORDER BY processed_at DESC, file_name
LIMIT 20
""")


,file_name,file_type,status,rows_loaded,processed_at,error_message
0,movimentacoes_202601.json,movimentacoes_json,SUCCESS,950,2026-05-14 03:31:02.394295+00:00,
1,movimentacoes_20260508.json,movimentacoes_json,SUCCESS,58,2026-05-14 02:43:15.629384+00:00,
2,movimentacoes_202605.json,movimentacoes_json,SUCCESS,698,2026-05-13 03:49:58.741188+00:00,
3,movimentacoes_20260507.json,movimentacoes_json,SUCCESS,56,2026-05-13 03:45:38.521220+00:00,
4,movimentacoes_20260506.json,movimentacoes_json,SUCCESS,70,2026-05-13 03:45:38.414125+00:00,
5,movimentacoes_20260505.json,movimentacoes_json,SUCCESS,19,2026-05-13 03:45:38.365975+00:00,
6,movimentacoes_20260504.json,movimentacoes_json,SUCCESS,35,2026-05-13 03:45:38.306205+00:00,
7,movimentacoes_20260503.json,movimentacoes_json,SUCCESS,45,2026-05-13 03:45:38.239647+00:00,
8,movimentacoes_202604.json,movimentacoes_json,SUCCESS,1564,2026-05-13 03:42:28.405076+00:00,
9,movimentacoes_202603.json,movimentacoes_json,SUCCESS,1097,2026-05-13 03:42:27.264362+00:00,


In [5]:
q("""
SELECT
    COUNT(*) AS total_registros,
    COUNT(*) FILTER (WHERE id_produto_servico IS NULL OR id_produto_servico = '') AS sem_id_produto,
    COUNT(*) FILTER (WHERE data_emissao IS NULL) AS sem_data_emissao,
    COUNT(*) FILTER (WHERE valor_total_liquido IS NULL) AS sem_valor_total
FROM silver.fato_movimentacoes
""")


,total_registros,sem_id_produto,sem_data_emissao,sem_valor_total
0,4947,0,0,0


## 3) Limpeza das bases (opcional)


In [6]:
# with engine.begin() as conn:
#     conn.execute(text("""
#         TRUNCATE TABLE
#             bronze.file_registry,
#             bronze.movimentacoes_raw,
#             bronze.produtos_servicos_raw,
#             silver.fato_movimentacoes,
#             silver.dim_produtos_servicos
#         RESTART IDENTITY CASCADE
#     """))
# print("Bases truncadas com sucesso.")
